# Student Scoring System (Rule-Based Intelligence)
This notebook computes rule-based scores and a readiness index from `students.csv`.

**Scores**
- Academic Performance Score (APS)
- Wellness & Wellbeing Score (WWS)
- Productivity & Time Management Score (PTMS)
- Career Readiness Score (CRS)
- Student Readiness Index (SRI)

**Categories**
- Green: SRI >= 85
- Blue: 75 <= SRI < 85
- Yellow: 60 <= SRI < 75
- Red: SRI < 60

In [ ]:
import pandas as pd
from fpdf import FPDF, XPos, YPos

INPUT_CSV = "students.csv"
OUTPUT_CSV = "students_scored.csv"
OUTPUT_PDF = "scoring_logic.pdf"

In [ ]:
def gpa_score(gpa):
    return (gpa / 10.0) * 100.0


def inverse_1_to_10(value):
    return (10.0 - value) / 9.0 * 100.0


def sleep_score(hours, target=7.0):
    return max(0.0, 100.0 - abs(hours - target) * 20.0)


def clamp_0_100(value):
    return max(0.0, min(100.0, value))


def category_from_sri(sri):
    if sri >= 85:
        return "Green"
    if sri >= 75:
        return "Blue"
    if sri >= 60:
        return "Yellow"
    return "Red"

In [ ]:
df = pd.read_csv(INPUT_CSV)

df["APS"] = (
    0.50 * gpa_score(df["gpa"])
    + 0.25 * df["attendance"]
    + 0.25 * df["assignments_completion"]
)

df["WWS"] = (
    0.40 * (df["mental_wellbeing"] * 10.0)
    + 0.30 * df["sleep_hours"].apply(sleep_score)
    + 0.30 * df["stress_level"].apply(inverse_1_to_10)
)

df["PTMS"] = (
    0.35 * (df["productivity_score"] * 10.0)
    + 0.25 * df["distractions"].apply(inverse_1_to_10)
    + 0.40 * df["engagement_score"]
)

df["CRS"] = (
    0.35 * (df["career_clarity"] * 10.0)
    + 0.35 * (df["skill_readiness"] * 10.0)
    + 0.30 * df["engagement_score"]
)

df["SRI"] = (
    0.35 * df["APS"]
    + 0.20 * df["WWS"]
    + 0.25 * df["PTMS"]
    + 0.20 * df["CRS"]
)

score_cols = ["APS", "WWS", "PTMS", "CRS", "SRI"]
df[score_cols] = df[score_cols].apply(lambda s: s.clip(0.0, 100.0))

df["Category"] = df["SRI"].apply(category_from_sri)
df.to_csv(OUTPUT_CSV, index=False)
df.head(5)

In [ ]:
sample_ids = ["S001", "S009", "S026"]
df.loc[df["student_id"].isin(sample_ids), ["student_id", "APS", "WWS", "PTMS", "CRS", "SRI", "Category"]]

In [ ]:
lines = [
    "Student Scoring System (Rule-Based Intelligence)",
    "",
    "Academic Performance Score (APS)",
    "- 50% GPA (scaled from 0-10 to 0-100)",
    "- 25% Attendance (%)",
    "- 25% Assignments Completion (%)",
    "",
    "Wellness & Wellbeing Score (WWS)",
    "- 40% Mental Wellbeing (1-10 mapped to 0-100)",
    "- 30% Sleep Score: 100 - |sleep_hours - 7| * 20",
    "- 30% Stress Level (1-10 inverted to 0-100)",
    "",
    "Productivity & Time Management Score (PTMS)",
    "- 35% Productivity Score (1-10 mapped to 0-100)",
    "- 25% Distractions (1-10 inverted to 0-100)",
    "- 40% Engagement Score (0-100)",
    "",
    "Career Readiness Score (CRS)",
    "- 35% Career Clarity (1-10 mapped to 0-100)",
    "- 35% Skill Readiness (1-10 mapped to 0-100)",
    "- 30% Engagement Score (0-100)",
    "",
    "Student Readiness Index (SRI)",
    "- 35% APS, 20% WWS, 25% PTMS, 20% CRS",
    "",
    "Categories",
    "- Green: SRI >= 85",
    "- Blue: 75 <= SRI < 85",
    "- Yellow: 60 <= SRI < 75",
    "- Red: SRI < 60",
]

pdf = FPDF()
pdf.set_margins(15, 15, 15)
pdf.set_auto_page_break(auto=True, margin=15)
pdf.set_compression(False)
pdf.add_page()
pdf.set_font("Helvetica", size=12)
line_height = 8
for line in lines:
    pdf.cell(0, line_height, text=line, new_x=XPos.LMARGIN, new_y=YPos.NEXT)

pdf.output(OUTPUT_PDF)
OUTPUT_PDF